# Reproducibility Notebook: Energy-Aware Sensing with RL

**Paper Revision for "Results in Engineering" Journal**

This notebook reproduces all results for the fixed RL agent that uses **persistence logic** (no oracle cheating).

## Key Fix
The original implementation had a methodological flaw: the agent could see ground-truth event flags even when sensors were OFF. This notebook implements the corrected version where:
- When sensor is **ON**: flag updates from ground truth
- When sensor is **OFF**: flag **persists** (stale value)

## Contents
1. Setup - Install dependencies, clone repository
2. Train - Run fixed training, show convergence
3. Synthetic Eval - Reproduce comparison table
4. MIT-BIH Eval - Real data validation
5. Figures - Detection vs Energy visualization

---
## 1. Setup

In [ ]:
# Install dependencies
!pip install -q numpy matplotlib wfdb

In [ ]:
# Clone repository (skip if already exists)
import os
if not os.path.exists('energy-aware-sensing-rl'):
    !git clone https://github.com/oussamaElallam/energy-aware-sensing-rl.git
os.chdir('energy-aware-sensing-rl')
print(f"Working directory: {os.getcwd()}")

In [ ]:
import sys
import random
import pickle
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import matplotlib.pyplot as plt

# Add framework to path
sys.path.insert(0, '.')
from framework.rl_env import HealthWearableEnv

print("\u2713 Setup complete!")

---
## 2. Train Q-Learning Agent (Fixed Persistence Logic)

### Verify the Fix
First, let's verify that the persistence logic is working correctly:

In [ ]:
# UNIT TEST: Verify persistence behavior
def test_persistence_logic():
    """Verify that event flags persist when sensors are OFF."""
    data = [
        {'arr_flag': 1, 'bp_flag': 0, 'fever_flag': 0},  # t=0: arrhythmia
        {'arr_flag': 0, 'bp_flag': 0, 'fever_flag': 0},  # t=1: no arrhythmia
        {'arr_flag': 0, 'bp_flag': 0, 'fever_flag': 0},
    ]
    
    env = HealthWearableEnv(data=data, max_time_steps=3)
    state = env.reset()
    assert state[2] == 1, "Initial arr_flag should be 1"
    
    # ECG OFF - flag should PERSIST
    next_state, _, _, _ = env.step(0)  # All sensors OFF
    assert next_state[2] == 1, "arr_flag should PERSIST when ECG OFF"
    print("\u2713 Test 1 PASSED: Flag persists when sensor is OFF")
    
    # Reset and test with ECG ON
    env = HealthWearableEnv(data=data, max_time_steps=3)
    env.reset()
    next_state, _, _, _ = env.step(4)  # ECG ON (0b100)
    assert next_state[2] == 0, "arr_flag should UPDATE when ECG ON"
    print("\u2713 Test 2 PASSED: Flag updates when sensor is ON")
    
    print("\n\u2705 Persistence logic verified!")

test_persistence_logic()

In [ ]:
# Q-Learning Training Function
def q_learning_train(
    env: HealthWearableEnv,
    episodes: int = 3000,
    gamma: float = 0.95,
    alpha_lr: float = 0.1,
    epsilon_start: float = 0.1,
    epsilon_min: float = 0.01,
) -> Tuple[Dict, List[float]]:
    """Train Q-learning agent with exponential epsilon decay."""
    Q = {}
    rewards = []
    epsilon = epsilon_start
    epsilon_decay = (epsilon_min / epsilon_start) ** (1 / episodes)
    
    for ep in range(episodes):
        s = env.reset()
        ep_r = 0.0
        done = False
        
        while not done:
            # Epsilon-greedy action selection
            if random.random() < epsilon:
                a = random.randrange(8)
            else:
                a = int(np.argmax([Q.get((s, b), 0.0) for b in range(8)]))
            
            s2, r, done, _ = env.step(a)
            
            # Q-learning update
            best_next = max(Q.get((s2, b), 0.0) for b in range(8))
            td_target = r + gamma * best_next
            td_error = td_target - Q.get((s, a), 0.0)
            Q[(s, a)] = Q.get((s, a), 0.0) + alpha_lr * td_error
            
            s = s2
            ep_r += r
        
        epsilon = max(epsilon_min, epsilon * epsilon_decay)
        rewards.append(ep_r)
        
        if (ep + 1) % 500 == 0:
            print(f"Episode {ep+1}/{episodes}, Avg Reward (last 50): {np.mean(rewards[-50:]):.2f}")
    
    return Q, rewards

In [ ]:
# Generate training data
STEPS = 12_000  # 16h at 5s cadence
rng = np.random.default_rng(0)

scenario = [
    {
        "arr_flag": int(rng.choice([0, 1], p=[0.7, 0.3])),
        "bp_flag": int(rng.choice([0, 1], p=[0.6, 0.4])),
        "fever_flag": int(rng.choice([0, 1], p=[0.8, 0.2])),
    }
    for _ in range(STEPS)
]

# Create environment
env = HealthWearableEnv(
    data=scenario,
    sensor_costs=[10, 4, 1],
    alpha=15.0,
    beta=0.008,
    lambda_risk=0.0,
    max_battery=400_000,
    max_time_steps=STEPS,
)

print(f"Training environment created: {STEPS} steps (~{STEPS*5/3600:.1f}h simulation)")

In [ ]:
# Train the agent
print("Training Q-learning agent with FIXED persistence logic...")
print("Hyperparameters: episodes=3000, gamma=0.95, alpha=0.1, epsilon: 0.1->0.01")
print()

Q, R = q_learning_train(env, episodes=3000)

print(f"\nTraining complete!")
print(f"Q-table size: {len(Q)} entries")
print(f"Final avg reward: {np.mean(R[-50:]):.2f}")

In [ ]:
# Plot training convergence
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(R, alpha=0.3, linewidth=0.5, label='Episode reward')
window = 50
moving_avg = np.convolve(R, np.ones(window)/window, mode='valid')
ax.plot(range(window-1, len(R)), moving_avg, color='red', linewidth=2, label=f'{window}-ep moving avg')

ax.set_xlabel('Episode')
ax.set_ylabel('Episode Reward')
ax.set_title('Training Convergence (Fixed Persistence Logic)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_convergence.png', dpi=150)
plt.show()

print("\u2713 Convergence plot saved to training_convergence.png")

In [ ]:
# Save Q-table
with open('q_table_fixed.pkl', 'wb') as f:
    pickle.dump(Q, f)
print("\u2713 Q-table saved to q_table_fixed.pkl")

---
## 3. Synthetic Evaluation

Compare policies on synthetic traces with multiple random seeds.

In [ ]:
# Define policies
SENSOR_COSTS = [10, 4, 1]

def always_on_policy(state):
    return 0b111

def periodic_5_30_policy(state):
    _, time_bucket, *_ = state
    return 0b100 if (time_bucket % 6) == 0 else 0b000

def heuristic_policy(state):
    battery, time_bucket, arr, bp, fever = state[:5]
    if arr or bp or fever:
        return 0b111  # All ON
    elif time_bucket % 6 == 0:
        return 0b100  # ECG only
    else:
        return 0b001  # Temp only

def rl_policy(state):
    qvals = [Q.get((state, a), 0.0) for a in range(8)]
    return int(np.argmax(qvals))

In [ ]:
def evaluate_policy(data, policy_fn):
    """Evaluate a policy on a trace."""
    env = HealthWearableEnv(
        data=data,
        sensor_costs=SENSOR_COSTS,
        alpha=15.0,
        beta=0.008,
        max_battery=400_000,
        max_time_steps=len(data),
    )
    
    state = env.reset()
    det_hits = det_total = energy_cost = 0
    
    while not env.done:
        action = policy_fn(state)
        next_state, _, done, _ = env.step(action)
        
        ecg_on = (action >> 2) & 1
        ppg_on = (action >> 1) & 1
        tmp_on = action & 1
        
        energy_cost += SENSOR_COSTS[0]*ecg_on + SENSOR_COSTS[1]*ppg_on + SENSOR_COSTS[2]*tmp_on
        
        if env.t <= len(data):
            gt = data[env.t - 1]
            for flag, on in [('arr_flag', ecg_on), ('bp_flag', ppg_on), ('fever_flag', tmp_on)]:
                if gt[flag]:
                    det_total += 1
                    if on:
                        det_hits += 1
        
        if done:
            break
        state = next_state
    
    det_rate = (det_hits / det_total * 100) if det_total > 0 else 0.0
    energy_mAh = energy_cost * 5 / 3600
    return det_rate, energy_mAh

In [ ]:
# Run evaluation with 10 seeds
N_SEEDS = 10
N_STEPS = 12_000

policies = [
    ("Always-On", always_on_policy),
    ("Periodic-5/30", periodic_5_30_policy),
    ("Heuristic", heuristic_policy),
    ("RL (Fixed)", rl_policy),
]

results = {name: {'det': [], 'energy': []} for name, _ in policies}

print(f"Evaluating on {N_SEEDS} synthetic traces...")
for seed in range(N_SEEDS):
    rng = np.random.default_rng(seed)
    data = [
        {'arr_flag': int(rng.random() < 0.10),
         'bp_flag': int(rng.random() < 0.30),
         'fever_flag': int(rng.random() < 0.10)}
        for _ in range(N_STEPS)
    ]
    
    for name, policy_fn in policies:
        det, energy = evaluate_policy(data, policy_fn)
        results[name]['det'].append(det)
        results[name]['energy'].append(energy)

print("Done!")

In [ ]:
# Display results table
print("\n" + "="*70)
print("SYNTHETIC TRACES RESULTS (16h simulation, 10 seeds)")
print("="*70)
print(f"{'Policy':<15} {'Detection (%)':<20} {'Energy (mAh)':<20} {'vs Always-On':<15}")
print("-"*70)

always_on_energy = np.mean(results["Always-On"]['energy'])

for name, _ in policies:
    det_mean = np.mean(results[name]['det'])
    det_std = np.std(results[name]['det'])
    energy_mean = np.mean(results[name]['energy'])
    energy_std = np.std(results[name]['energy'])
    reduction = (1 - energy_mean / always_on_energy) * 100
    
    print(f"{name:<15} {det_mean:>6.1f} \u00b1 {det_std:<6.1f}     {energy_mean:>6.1f} \u00b1 {energy_std:<6.1f}     {reduction:>+6.1f}%")

print("="*70)

---
## 4. MIT-BIH Real Data Evaluation

Evaluate on real ECG data from MIT-BIH Arrhythmia Database.

**Note**: MIT-BIH is ECG-only, so `bp_flag` and `fever_flag` are always 0.

In [ ]:
import wfdb

# MIT-BIH parameters
FS_MITBIH = 360  # Hz
WINDOW_SECONDS = 5
SAMPLES_PER_WINDOW = WINDOW_SECONDS * FS_MITBIH  # 1800 samples
ABNORMAL_SYMBOLS = ['L', 'R', 'A', 'V', 'F', '/', 'f', 'j', 'a', 'S', 'E']

def convert_record_to_events(record_path):
    """Convert MIT-BIH record to event trace."""
    ann = wfdb.rdann(record_path, 'atr')
    record = wfdb.rdrecord(record_path)
    
    total_samples = record.sig_len
    n_windows = total_samples // SAMPLES_PER_WINDOW
    
    events = []
    for w in range(n_windows):
        start = w * SAMPLES_PER_WINDOW
        end = (w + 1) * SAMPLES_PER_WINDOW
        
        arr_flag = 0
        for i, sample in enumerate(ann.sample):
            if start <= sample < end:
                if ann.symbol[i] in ABNORMAL_SYMBOLS:
                    arr_flag = 1
                    break
        
        events.append({'arr_flag': arr_flag, 'bp_flag': 0, 'fever_flag': 0})
    
    return events

In [ ]:
# Download and evaluate sample records
sample_records = ['100', '101', '102', '103', '104', '105']
mitbih_results = []

os.makedirs('mitbih_data', exist_ok=True)

for record_id in sample_records:
    print(f"Processing record {record_id}...")
    try:
        record_path = f'mitbih_data/{record_id}'
        if not os.path.exists(f'{record_path}.dat'):
            wfdb.dl_database('mitdb', 'mitbih_data', records=[record_id])
        
        events = convert_record_to_events(record_path)
        n_anomalies = sum(1 for e in events if e['arr_flag'])
        print(f"  {len(events)} windows, {n_anomalies} ({n_anomalies/len(events)*100:.1f}%) with arrhythmia")
        
        for name, policy_fn in [("Always-On", always_on_policy), 
                                 ("Heuristic", heuristic_policy),
                                 ("RL-Fixed", rl_policy)]:
            det, energy = evaluate_policy(events, policy_fn)
            mitbih_results.append({
                'record': record_id,
                'policy': name,
                'detection': det,
                'energy': energy,
            })
    except Exception as e:
        print(f"  Error: {e}")

In [ ]:
# Display MIT-BIH results
print("\n" + "="*60)
print("MIT-BIH REAL DATA RESULTS")
print("="*60)

for policy in ["Always-On", "Heuristic", "RL-Fixed"]:
    policy_results = [r for r in mitbih_results if r['policy'] == policy]
    if policy_results:
        det_rates = [r['detection'] for r in policy_results]
        energies = [r['energy'] for r in policy_results]
        print(f"\n{policy}:")
        print(f"  Detection: {np.mean(det_rates):.1f}% \u00b1 {np.std(det_rates):.1f}%")
        print(f"  Energy:    {np.mean(energies):.2f} \u00b1 {np.std(energies):.2f} mAh")

---
## 5. Figures

In [ ]:
# Detection vs Energy Trade-off Plot
fig, ax = plt.subplots(figsize=(8, 6))

colors = {'Always-On': 'red', 'Periodic-5/30': 'orange', 'Heuristic': 'blue', 'RL (Fixed)': 'green'}
markers = {'Always-On': 's', 'Periodic-5/30': '^', 'Heuristic': 'D', 'RL (Fixed)': 'o'}

for name, _ in policies:
    det_mean = np.mean(results[name]['det'])
    det_std = np.std(results[name]['det'])
    energy_mean = np.mean(results[name]['energy'])
    energy_std = np.std(results[name]['energy'])
    
    ax.errorbar(energy_mean, det_mean, 
                xerr=energy_std, yerr=det_std,
                fmt=markers[name], markersize=12,
                color=colors[name], label=name,
                capsize=5, capthick=2, elinewidth=2)

ax.set_xlabel('Energy Consumption (mAh)', fontsize=12)
ax.set_ylabel('Detection Rate (%)', fontsize=12)
ax.set_title('Detection vs Energy Trade-off (Synthetic 16h)', fontsize=14)
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 280)
ax.set_ylim(0, 110)

plt.tight_layout()
plt.savefig('detection_vs_energy.png', dpi=150)
plt.show()

print("\u2713 Figure saved to detection_vs_energy.png")

---
## Summary

This notebook demonstrates:
1. **Fix verified**: Persistence logic correctly prevents oracle cheating
2. **Training**: Q-learning converges with fixed environment
3. **Synthetic evaluation**: RL policy achieves good detection with significant energy savings
4. **Real data**: MIT-BIH validation confirms results

**Key insight**: Without oracle access, detection rate is lower but the policy is methodologically correct and honest.